# Stage 6e Part B: baseline hyperparameter search results (K=400 dense search)

Companion notebook for `scripts/6e_baseline_hp_search_k400.py`. Presents that script's
already-completed dense K=400-only baseline hyperparameter search (100 configs x 5 seeds =
500 runs) as clean tables for transfer into the manuscript's supplemental materials
(`supplement.tex`) - this is the search behind reviewer Point 5 (baseline adequacy):
with a real, dense search done (not a sparse incidental sample), does the optimized-
vs-baseline gap survive under the best baseline we could find at matched training budget?

This is a separate notebook from `6e_baseline_convergence.ipynb` (Part A: the baseline's
own training-loss convergence curve) - that one is left untouched.

In [ ]:
import os, sys
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import json
import glob
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

BASELINE_HP_DIR = os.path.join(PROJECT_ROOT, "data", "SI_results", "baseline_hp")
K400_DIR = os.path.join(BASELINE_HP_DIR, "k400_search")
GAP_DIR = os.path.join(BASELINE_HP_DIR, "gap_persistence")

# Same per-scenario weighting scripts/6e_baseline_hp_search_k400.py's own scoring uses
# (EVAL_WEIGHTS there) - "All" is reported for reference but excluded from the score itself.
EVAL_WEIGHTS = {"Tier 1": 7, "Tier 2": 5, "DECK": 2, "CS3": 2}

def weighted_mean(eval_scores):
    return sum(EVAL_WEIGHTS[k] * eval_scores[k] for k in EVAL_WEIGHTS) / sum(EVAL_WEIGHTS.values())

### Table 1 - dense K=400 search, ranked (top 20 of 100 configs)

Each config was run at 5 seeds; `mean_score`/`std_score` aggregate across seeds using the
search script's own per-run `score` field (weighted mean of Tier 1/Tier 2/DECK/CS3, matching
`EVAL_WEIGHTS` above). `All` is shown for reference only, not part of the ranking score.

In [ ]:
files = glob.glob(os.path.join(K400_DIR, "search_cfg*_seed*_result.json"))
raw_rows = [json.load(open(f)) for f in files]

df = pd.DataFrame(raw_rows)
eval_df = pd.json_normalize(df["eval_scores"]).rename(columns={"Tier 1": "Tier1", "Tier 2": "Tier2"})
df = pd.concat([df.drop(columns=["eval_scores"]), eval_df], axis=1)

df_search = df.groupby("config_idx").agg(
    lr=("lr", "first"), weight_decay=("weight_decay", "first"), K=("K", "first"),
    mean_score=("score", "mean"), std_score=("score", "std"), n_seeds=("seed", "count"),
    all_stable=("stable", "all"),
    Tier1=("Tier1", "mean"), Tier2=("Tier2", "mean"), DECK=("DECK", "mean"),
    CS3=("CS3", "mean"), All=("All", "mean"),
).reset_index().sort_values("mean_score").reset_index(drop=True)

print(f"{len(df_search)} configs total (from {len(raw_rows)} config x seed runs)")
df_search.head(20)

### Table 2 - reference comparison

The dense search's winner against the two other baseline configs discussed in
`REVISIONS.md`'s Stage 6e narrative: the pipeline's original copied-over default
(`lr=0.05, weight_decay=0.01`), and the config transferred from the K=1600 search winner
(`lr=0.1329, weight_decay=0.0` - shown there to *not* transfer well down to K=400). Both
reference configs' raw per-seed evaluations are on disk at
`data/SI_results/baseline_hp/gap_persistence/gap_{original,tuned}_seed*_result.json`;
their `weighted_mean` is computed here identically to the search's own scoring.

In [ ]:
best = json.load(open(os.path.join(K400_DIR, "best_baseline_config_K400.json")))

ref_rows = []
for name, label in [("original", "Copied-over default"), ("tuned", "K=1600-transferred")]:
    gap_files = sorted(glob.glob(os.path.join(GAP_DIR, f"gap_{name}_seed*_result.json")))
    gap_rows = [json.load(open(f)) for f in gap_files]
    scores = [weighted_mean(r["eval_scores"]) for r in gap_rows]
    ref_rows.append({
        "label": label, "lr": gap_rows[0]["lr"], "weight_decay": gap_rows[0]["weight_decay"],
        "K": gap_rows[0]["K"], "mean_score": float(np.mean(scores)), "n_seeds": len(scores),
    })
ref_rows.append({
    "label": "K=400 dense-search winner", "lr": best["config"]["lr"],
    "weight_decay": best["config"]["weight_decay"], "K": best["config"]["K"],
    "mean_score": best["mean_score"], "n_seeds": best["n_seeds"],
})

df_reference = pd.DataFrame(ref_rows)
df_reference["pct_vs_default"] = (
    (df_reference["mean_score"].iloc[0] - df_reference["mean_score"]) / df_reference["mean_score"].iloc[0] * 100
)
df_reference

### Export (for direct transfer into the supplement)

In [ ]:
df_search.to_csv(os.path.join(K400_DIR, "baseline_k400_search_full_table.csv"), index=False)
df_search.head(20).to_csv(os.path.join(K400_DIR, "baseline_k400_search_top20_table.csv"), index=False)
df_reference.to_csv(os.path.join(K400_DIR, "baseline_reference_comparison_table.csv"), index=False)
print("Wrote 3 CSVs to", K400_DIR)